# 04 – Phân cụm: GMM và K-Means (#21)

Notebook chỉ gọi hàm trong `src/cluster.py`. Dữ liệu: `clean` → `add_derived_features` → chia 7:3 có phân tầng, seed 0 → `build_preprocessor` → PCA 11 thành phần (11 = `n_components_reduced`). Preprocessor, PCA, GMM, K-Means **chỉ fit trên train**; validation chỉ transform / predict. Phân cụm **bỏ qua nhãn**; nhãn chỉ dùng để đánh giá cụm. Hình lưu vào `results/figures/` (`cluster_*.png`), mô hình lưu vào `data/models/` (gitignore) để tái lập kết quả mục 4.3. Mô-đun 2 tầng (mục 5) **không dùng mô hình này**: nó fit GMM riêng trên khối A (LIMIT_BAL, SEX, EDUCATION, MARRIAGE, AGE) và chỉ dùng lại `select_k_gmm` / `fit_gmm` / `save_models`.

**Lưu ý phương pháp.** GMM covariance đầy đủ trên PCA nhạy với khởi tạo và dễ suy biến (một thành phần co về không gian con rời rạc do one-hot SEX/EDUCATION/MARRIAGE và PAY = 0). Vì vậy dùng `reg_covar = 1e-2` và cùng `n_init = 5` cho chọn k lẫn fit cuối, chọn k theo **độ ổn định giữa 5 seed** chứ không theo min BIC của một seed, và có một bước kiểm tra suy biến (mục 3b).

In [1]:
import sys, time
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.config import load_config
from src.data import load_raw, TARGET
from src.preprocess import clean, add_derived_features, make_splits
from src import reduce as rd
from src import cluster as cl

pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 30)
FIG = cl.FIG_DIR
cfg = load_config()
K_RANGE = cfg['clustering']['k_range']          # [2, 10]
K_PCA = cfg['n_components_reduced']             # 11
SEEDS = cfg['seeds']
T0 = time.time()

## 0. Dữ liệu, PCA 11 chiều

In [2]:
df = add_derived_features(clean(load_raw()))
y = df[TARGET]
tr_idx, va_idx = make_splits(y, [0], {'7:3': 0.3})[('7:3', 0)]
df_tr, df_va = df.loc[tr_idx], df.loc[va_idx]   # index là nhãn dòng gốc -> dùng .loc
y_tr, y_va = y.loc[tr_idx].to_numpy(), y.loc[va_idx].to_numpy()

pre, X_tr, X_va = rd.pca_input(df_tr, df_va)
pca = rd.fit_pca(X_tr, n_components=K_PCA)
Z_tr, Z_va = pca.transform(X_tr), pca.transform(X_va)
print('Z_train', Z_tr.shape, '| Z_val', Z_va.shape, '| phương sai giữ lại =', round(pca.explained_variance_ratio_.sum(), 4))
pd.DataFrame({'số mẫu': [len(y_tr), len(y_va)], 'tỉ lệ vỡ nợ': [y_tr.mean(), y_va.mean()]}, index=['train', 'validation']).round(4)

EDUCATION = 0 -> 4: 14 mẫu
EDUCATION = 5 -> 4: 280 mẫu
EDUCATION = 6 -> 4: 51 mẫu
MARRIAGE = 0 -> 3: 54 mẫu
Loại bản ghi trùng hoàn toàn: 35 dòng (còn 29965)
Z_train (20975, 11) | Z_val (8990, 11) | phương sai giữ lại = 0.818


,số mẫu,tỉ lệ vỡ nợ
train,20975,0.2213
validation,8990,0.2212


## 1. Chọn số cụm

**GMM** (`covariance_type='full'`, `reg_covar = 1e-2`, `n_init = 5`): với mỗi k = 2…10 và mỗi seed 0…4 fit GMM, ghi BIC/AIC và phân hoạch. Độ ổn định = ARI trung bình giữa các cặp seed (1 = mọi seed ra cùng cụm). **Quy tắc chọn k:** trong các k có ARI trung bình ≥ 0,8 (ổn định), bỏ k = 2 (ổn định tầm thường), lấy k có BIC trung bình nhỏ nhất; không k nào đạt thì kết luận GMM không có k ổn định. **K-Means** (`n_init = 10`): inertia (elbow) và silhouette; silhouette tính trên mẫu con cố định 5000 dòng (seed 0) vì độ phức tạp O(n²), cùng mẫu con cho mọi k.

In [3]:
t = time.time()
stab_gmm = cl.k_stability(Z_tr, K_RANGE, SEEDS)      # reg_covar, n_init mặc định = cl.REG_COVAR, cl.N_INIT_GMM
print(f'GMM: {time.time() - t:.0f} s | reg_covar = {cl.REG_COVAR}, n_init = {cl.N_INIT_GMM}, seeds = {SEEDS}')
tab_gmm = stab_gmm['summary']
k_gmm = cl.choose_k_stable(tab_gmm)
k_minbic_mean = int(tab_gmm['bic_mean'].idxmin())
print('k chọn (ổn định, BIC trung bình nhỏ nhất) =', k_gmm, '| k theo min BIC trung bình =', k_minbic_mean,
      '| k theo min AIC trung bình =', int(stab_gmm['aic'].mean(axis=1).idxmin()))
tab_gmm.round(3)

GMM: 235 s | reg_covar = 0.01, n_init = 5, seeds = [0, 1, 2, 3, 4]
k chọn (ổn định, BIC trung bình nhỏ nhất) = 4 | k theo min BIC trung bình = 10 | k theo min AIC trung bình = 10


,bic_mean,bic_min,bic_max,ari_mean,gain_%
k,,,,,
2,684894.436,684893.168,684895.997,0.999,NaN
3,573841.041,573840.988,573841.062,1.000,16.215
4,516679.196,516679.196,516679.196,1.000,9.961
5,475325.905,451234.736,489051.367,0.698,8.004
6,439059.016,420110.226,454596.751,0.384,7.630
7,427338.295,419994.196,431817.672,0.290,2.670
8,400900.495,386268.199,416884.836,0.309,6.187
9,382865.865,374567.048,395201.564,0.347,4.499
10,369629.404,348603.262,386647.471,0.336,3.457


In [4]:
cl.plot_k_gmm(stab_gmm, FIG / 'cluster_gmm_bic_aic.png', k_chosen=k_gmm)
tab_km = cl.select_k_kmeans(Z_tr, K_RANGE, seed=0)
k_sil, k_elbow = int(tab_km['silhouette'].idxmax()), cl.elbow_k(tab_km, 'inertia')
cl.plot_k_kmeans(tab_km, FIG / 'cluster_kmeans_elbow_silhouette.png', k_chosen=k_sil)
print('k theo max silhouette =', k_sil, '| khuỷu inertia =', k_elbow)
tab_km.round(3)

k theo max silhouette = 2 | khuỷu inertia = 5


,inertia,silhouette
k,,
2,425451.650,0.263
3,372793.460,0.223
4,335194.514,0.189
5,311843.689,0.162
6,295901.585,0.167
7,278334.610,0.173
8,265623.080,0.166
9,253896.178,0.171
10,240039.254,0.173


![](../results/figures/cluster_gmm_bic_aic.png)

![](../results/figures/cluster_kmeans_elbow_silhouette.png)

**Nhận xét – GMM.** BIC trung bình giảm gần như liên tục từ 684 894 (k = 2) xuống 369 629 (k = 10), mỗi bước vẫn giảm 2,7–16%, và cực tiểu BIC/AIC trung bình nằm ở **mốc cuối** k = 10: BIC không "dừng giảm" nên min BIC không chọn được k. Đồng thời dải min–max qua 5 seed rộng ra nhanh từ k = 5 (k = 10: 348 603 đến 386 647), và ARI trung bình giữa các seed là 0,999 / 1,000 / 1,000 với k = 2 / 3 / 4 nhưng tụt còn 0,698 ở k = 5 và 0,29–0,38 ở k = 6…10: từ k = 5 trở lên mỗi seed ra một phân hoạch khác nhau (EM kẹt ở các cực trị địa phương khác nhau; ở k lớn, các thành phần vẫn sụp vào hướng one-hot (SEX/EDUCATION/MARRIAGE) dù `reg_covar = 1e-2` (kiểm tra ở k = 9: min eigenvalue ≈ 0,010, nhiều cụm thuần 100% theo một biến nhân khẩu), đây cũng là lý do mất ổn định). Theo quy tắc ở trên (ARI ≥ 0,8, bỏ k = 2, BIC trung bình nhỏ nhất), **chọn k = 4** (BIC trung bình 516 679, mọi seed cùng một giá trị). Phải nói rõ: k = 4 là k ổn định **lớn nhất**, không phải điểm BIC tối ưu; BIC vẫn còn giảm tiếp ở k lớn hơn nhưng các k đó không tái lập được giữa các seed nên không đáng tin. BIC trung bình không có khuỷu rõ (mức giảm mỗi bước 16,2% → 10,0% → 8,0% → 7,6% rồi 2,7–6,2%), nên đây là lựa chọn dựa trên độ ổn định. Kết quả không phụ thuộc ngưỡng: mọi ngưỡng τ trong khoảng 0,70–0,99 đều cho k = 4.

**Nhận xét – K-Means.** Silhouette cao nhất ở k = 2 (0,263) rồi tụt còn 0,16–0,22 khi k ≥ 3; khuỷu inertia ở k = 5. Chọn chính **k = 2 theo silhouette**; k = 5 (khuỷu) và k = 4 (bằng GMM) giữ làm đối chứng ở các mục sau.

## 2. Chọn k theo từng seed (0..4)

Cùng Z_train (split và PCA không đổi), chỉ đổi seed khởi tạo. GMM: k có BIC nhỏ nhất và `BIC_min` của từng seed (lấy từ bảng ở mục 1); K-Means: k theo silhouette và theo khuỷu.

In [5]:
rows = []
for s in SEEDS:
    km = cl.select_k_kmeans(Z_tr, K_RANGE, seed=s)
    rows.append({'seed': s, 'k_GMM(min BIC)': int(stab_gmm['bic'][s].idxmin()), 'k_GMM(min AIC)': int(stab_gmm['aic'][s].idxmin()),
                 'BIC_min': stab_gmm['bic'][s].min(), 'k_KMeans(max silhouette)': int(km['silhouette'].idxmax()),
                 'k_KMeans(khuỷu)': cl.elbow_k(km, 'inertia')})
stab = pd.DataFrame(rows).set_index('seed')
stab.round(1)

,k_GMM(min BIC),k_GMM(min AIC),BIC_min,k_KMeans(max silhouette),k_KMeans(khuỷu)
seed,,,,,
0,10,10,364413.2,2,5
1,10,10,370996.5,2,5
2,10,10,348603.3,2,4
3,9,9,374567.0,2,4
4,9,9,378724.7,2,5


**Nhận xét.** K-Means ổn định: silhouette luôn chọn k = 2, khuỷu inertia nằm ở k = 4–5. GMM theo từng seed vẫn **không ổn định ở k lớn**: min BIC rơi vào k = 10 ở seed 0, 1, 2 và k = 9 ở seed 3, 4 (AIC trùng BIC); `BIC_min` dao động từ 348 603 đến 378 725 giữa các seed. Ngoài cực trị địa phương của EM, ở k lớn, các thành phần vẫn sụp vào hướng one-hot (SEX/EDUCATION/MARRIAGE) dù `reg_covar = 1e-2` (kiểm tra ở k = 9: min eigenvalue ≈ 0,010, nhiều cụm thuần 100% theo một biến nhân khẩu), đây cũng là lý do mất ổn định. Đó là lý do không chọn k theo min BIC của một seed (sẽ ra 9 hoặc 10 tùy seed) mà theo độ ổn định ở mục 1 (k = 4).

## 3. Fit mô hình cuối, chất lượng cụm (không dùng nhãn)

GMM với k đã chọn ở mục 1 (cùng `reg_covar`, `n_init` như lúc chọn, seed 0). K-Means ở **k theo silhouette** (chọn chính) và, để so sánh công bằng với GMM, ở k khuỷu và ở k bằng k của GMM. Cụm được đánh số lại theo kích thước giảm dần (cụm 0 lớn nhất). Độ đo: silhouette (mẫu con 5000, cao tốt), Davies–Bouldin (thấp tốt), Calinski–Harabasz (cao tốt).

In [6]:
assert k_gmm is not None, 'không có k GMM ổn định: cần xem lại mục 1'
gmm = cl.fit_gmm(Z_tr, k_gmm, seed=0)
km_main = cl.fit_kmeans(Z_tr, k_sil, seed=0)
km_elb = cl.fit_kmeans(Z_tr, k_elbow, seed=0)
km_same = cl.fit_kmeans(Z_tr, k_gmm, seed=0)

models = {f'GMM k={k_gmm}': gmm, f'K-Means k={k_sil} (silhouette)': km_main,
          f'K-Means k={k_elbow} (khuỷu)': km_elb, f'K-Means k={k_gmm} (= GMM)': km_same}
raw_tr = {n: m.predict(Z_tr) for n, m in models.items()}
raw_va = {n: m.predict(Z_va) for n, m in models.items()}
c_tr, c_va = {}, {}
for n in models:
    c_tr[n], mp = cl.relabel_by_size(raw_tr[n])
    c_va[n] = cl.apply_mapping(raw_va[n], mp)      # cùng bản đồ đánh số; báo lỗi nếu cụm không có mẫu train

qual = pd.DataFrame({n: cl.quality_metrics(Z_tr, c_tr[n], seed=0) for n in models}).T
qual

,silhouette,davies_bouldin,calinski_harabasz
GMM k=4,0.101522,2.115821,3008.290583
K-Means k=2 (silhouette),0.262590,1.555666,6937.956748
K-Means k=5 (khuỷu),0.162024,1.809308,4275.947629
K-Means k=4 (= GMM),0.188806,1.764745,4817.370868


**Nhận xét.** GMM k = 4 có silhouette 0,102, Davies–Bouldin 2,12 và Calinski–Harabasz 3 008: kém mọi cấu hình K-Means ở cả ba độ đo: cùng k = 4 (0,189 / 1,76 / 4 817), k = 5 (0,162 / 1,81 / 4 276) và k = 2 (0,263 / 1,56 / 6 938). Các cụm GMM chồng lấn nhiều theo khoảng cách Euclid trong 11 chiều PCA, nhưng GMM full-covariance chia cả theo hình dạng/hướng phương sai nên ba độ đo dựa trên khoảng cách không hoàn toàn công bằng với nó. Silhouette và CH vốn thiên về k nhỏ nên so K-Means k = 2 với k = 4 không cùng điều kiện.

## 3b. Kiểm tra suy biến của GMM

Mỗi thành phần: trọng số, eigenvalue nhỏ/lớn nhất của covariance (so với `reg_covar`), log định thức, và độ thuần của cụm theo SEX / EDUCATION / MARRIAGE (tỉ lệ giá trị phổ biến nhất; 1 = cụm chỉ có một giá trị) và tỉ lệ dòng có mọi PAY_1..PAY_6 = 0. Đối chứng: cùng k, `reg_covar = 1e-6` (mặc định sklearn).

In [7]:
PAYS = [f'PAY_{i}' for i in range(1, 7)]
lab_cols = df_tr[['SEX', 'EDUCATION', 'MARRIAGE'] + PAYS]
deg = cl.gmm_degeneracy(gmm, Z_tr, lab_cols)
print(f'reg_covar = {gmm.reg_covar:g} | tỉ lệ dòng train có PAY_1..PAY_6 đều = 0: {(df_tr[PAYS] == 0).all(axis=1).mean():.3f}')
display(deg.sort_values('weight', ascending=False).round(4))
print('min_eig nhỏ nhất của mọi thành phần =', f"{deg['min_eig'].min():.2e}", '| gấp', round(deg['min_eig'].min() / gmm.reg_covar, 2), 'lần reg_covar')
print('số thành phần có min_eig < 2 x reg_covar:', int((deg['min_eig'] < 2 * gmm.reg_covar).sum()), f'/ {len(deg)}')
print('số cụm thuần (>= 0,99) theo SEX / EDUCATION / MARRIAGE; số cụm có pay_all0 >= 0,99 hoặc <= 0,01:',
      int((deg['pure_SEX'] >= 0.99).sum()), '/', int((deg['pure_EDUCATION'] >= 0.99).sum()), '/', int((deg['pure_MARRIAGE'] >= 0.99).sum()), ';',
      int(((deg['pay_all0'] >= 0.99) | (deg['pay_all0'] <= 0.01)).sum()))
gmm_eps = cl.fit_gmm(Z_tr, k_gmm, seed=0, reg_covar=1e-6)
deg_eps = cl.gmm_degeneracy(gmm_eps, Z_tr, lab_cols)
print('Đối chứng reg_covar = 1e-6: min_eig thấp nhất =', f"{deg_eps['min_eig'].min():.2e}",
      '| số thành phần min_eig < 1e-5:', int((deg_eps['min_eig'] < 1e-5).sum()),
      '| BIC =', round(gmm_eps.bic(Z_tr)), '(reg 1e-2:', round(gmm.bic(Z_tr)), ')')
deg_eps.sort_values('weight', ascending=False).round(4)

reg_covar = 0.01 | tỉ lệ dòng train có PAY_1..PAY_6 đều = 0: 0.330


,weight,min_eig,max_eig,logdet,size,pure_SEX,pure_EDUCATION,pure_MARRIAGE,pay_all0
thành phần,,,,,,,,,
3,0.4167,0.0132,8.9403,-2.5481,8698,0.6350,0.4784,0.5160,0.0283
2,0.3022,0.0102,2.4440,-17.9692,6381,0.6045,0.5317,0.5620,0.9914
1,0.2534,0.0127,6.0105,-3.2486,5315,0.5599,0.5390,0.5486,0.0218
0,0.0277,0.3291,37.4115,6.6178,581,0.6540,0.5439,0.4647,0.3976


min_eig nhỏ nhất của mọi thành phần = 1.02e-02 | gấp 1.02 lần reg_covar
số thành phần có min_eig < 2 x reg_covar: 3 / 4
số cụm thuần (>= 0,99) theo SEX / EDUCATION / MARRIAGE; số cụm có pay_all0 >= 0,99 hoặc <= 0,01: 0 / 0 / 0 ; 1


Đối chứng reg_covar = 1e-6: min_eig thấp nhất = 8.19e-05 | số thành phần min_eig < 1e-5: 0 | BIC = 471634 (reg 1e-2: 516679 )


,weight,min_eig,max_eig,logdet,size,pure_SEX,pure_EDUCATION,pure_MARRIAGE,pay_all0
thành phần,,,,,,,,,
1,0.4207,0.0031,9.1145,-4.9141,8841,0.6318,0.4869,0.5202,0.0649
0,0.2759,0.0001,2.4520,-27.6465,5821,0.6037,0.5341,0.5582,0.9993
3,0.2757,0.0024,6.3481,-5.5539,5732,0.5680,0.5340,0.5482,0.0518
2,0.0277,0.3191,37.4015,6.5109,581,0.6540,0.5439,0.4647,0.3976


**Nhận xét – suy biến.** Với `reg_covar = 1e-2`, không thành phần nào suy biến hoàn toàn: eigenvalue nhỏ nhất là 0,0102 (thành phần 2 (= cụm 1 sau khi đánh số lại theo kích thước), cụm 99,1% dòng có mọi PAY = 0), log định thức thấp nhất −18,0; so với đối chứng `reg_covar = 1e-6` thì thành phần PAY = 0 co xuống min_eig = 8,2e-5, log định thức −27,6 (BIC 471 634 thấp hơn 516 679, nhiều khả năng do phương sai gần 0 này đẩy log-likelihood lên chứ không phải mô hình tốt hơn). Tuy nhiên **3/4 thành phần vẫn có min_eig trong khoảng 1,0–1,3 lần `reg_covar`** (0,0102 / 0,0127 / 0,0132): tức covariance vẫn bị "sàn" regularization giữ lại ở các hướng rời rạc (one-hot SEX/EDUCATION/MARRIAGE và PAY = 0 trong PCA), `reg_covar` chỉ ngăn nó sụp hẳn, không xóa hướng đó. Độ thuần theo SEX / EDUCATION / MARRIAGE của mọi cụm chỉ 0,46–0,65 (gần tỉ lệ của toàn tập: nữ 0,61, đại học 0,47, có gia đình 0,46), tức **không cụm nào tách theo nhân khẩu**; riêng PAY: cụm 1 có 99,1% dòng mọi PAY = 0 và hai cụm lớn còn lại chỉ 2–3%. Vậy nhóm PAY = 0 (dùng tín dụng quay vòng, 33,0% dòng) là hành vi có thật; điều mang tính "hiện vật" là hình dạng hẹp (phương sai gần 0) của cụm do điểm khối PAY = 0 tạo ra; cách đọc an toàn là các cụm GMM chủ yếu phân theo **hành vi trả nợ** và chồng lấn nhiều, như silhouette thấp ở mục 3 cho thấy.

## 4. Quan hệ cụm – nhãn

Bảng chéo cụm × nhãn trên train: kích thước, tỉ trọng, tỉ lệ vỡ nợ từng cụm của GMM; kiểm định chi-bình phương (H0: cụm và nhãn độc lập), Cramér's V, ARI, NMI. Vì nhãn không được dùng khi phân cụm nên ARI/NMI thấp là bình thường; ý nghĩa nằm ở chỗ tỉ lệ vỡ nợ có khác nhau giữa các cụm hay không.

In [8]:
G = f'GMM k={k_gmm}'
rep_gmm = cl.cluster_label_report(c_tr[G], y_tr)
rep_gmm['table'].round(4)

,size,share,n_default,default_rate
cụm,,,,
0,8698,0.4147,1488,0.1711
1,6381,0.3042,669,0.1048
2,5315,0.2534,2405,0.4525
3,581,0.0277,79,0.1360


In [9]:
def summary(reps):
    return pd.DataFrame({n: {k: v for k, v in r.items() if k != 'table'} for n, r in reps.items()}).T

reps_tr = {n: cl.cluster_label_report(c_tr[n], y_tr) for n in models}
reps_va = {n: cl.cluster_label_report(c_va[n], y_va) for n in models}
print('TRAIN'); display(summary(reps_tr).round(4))
print('VALIDATION (gán cụm bằng predict, không fit lại)'); display(summary(reps_va).round(4))

TRAIN


,chi2,dof,p,cramers_v,ari,nmi
GMM k=4,2302.8887,3.0,0.0,0.3313,0.0557,0.0601
K-Means k=2 (silhouette),37.1427,1.0,0.0,0.0421,-0.0134,0.0016
K-Means k=5 (khuỷu),2574.8403,4.0,0.0,0.3504,0.0437,0.0497
K-Means k=4 (= GMM),2654.7737,3.0,0.0,0.3558,0.0787,0.0613


VALIDATION (gán cụm bằng predict, không fit lại)


,chi2,dof,p,cramers_v,ari,nmi
GMM k=4,925.5073,3.0,0.0000,0.3209,0.0536,0.0567
K-Means k=2 (silhouette),15.9118,1.0,0.0001,0.0421,-0.0129,0.0016
K-Means k=5 (khuỷu),1141.3477,4.0,0.0000,0.3563,0.0458,0.0513
K-Means k=4 (= GMM),1224.6217,3.0,0.0000,0.3691,0.0858,0.0655


In [10]:
rv = reps_va[G]['table']
cmp_tab = pd.DataFrame({'size_tr': rep_gmm['table']['size'], 'rate_tr': rep_gmm['table']['default_rate'],
                        'share_tr': rep_gmm['table']['share'], 'size_va': rv['size'], 'share_va': rv['share'],
                        'rate_va': rv['default_rate']})
cmp_tab['chênh_tỉ_lệ'] = cmp_tab['rate_va'] - cmp_tab['rate_tr']
print('Tương quan Pearson giữa tỉ lệ vỡ nợ từng cụm train và validation:', round(cmp_tab['rate_tr'].corr(cmp_tab['rate_va']), 3))
cmp_tab.round(3)

Tương quan Pearson giữa tỉ lệ vỡ nợ từng cụm train và validation: 0.995


,size_tr,rate_tr,share_tr,size_va,share_va,rate_va,chênh_tỉ_lệ
cụm,,,,,,,
0,8698,0.171,0.415,3802,0.423,0.176,0.005
1,6381,0.105,0.304,2693,0.300,0.103,-0.002
2,5315,0.452,0.253,2239,0.249,0.446,-0.006
3,581,0.136,0.028,256,0.028,0.168,0.032


**Nhận xét.** Tỉ lệ vỡ nợ của 4 cụm GMM là 10,5% (cụm 1) / 13,6% (cụm 3) / 17,1% (cụm 0) / 45,3% (cụm 2) quanh mức chung 22,1%. Kiểm định chi-bình phương: χ² = 2 303, dof = 3, p làm tròn về 0 (bảng chỉ hiện 4 chữ số), Cramér's V = 0,331, tức cụm và nhãn **không độc lập** và mức liên hệ vừa. ARI = 0,056 và NMI = 0,060 gần 0: cụm **không tái tạo lại nhãn**; cụm vỡ nợ cao nhất (cụm 2) vẫn có 55% là khách không vỡ nợ. Trên validation (gán cụm bằng `predict`) V = 0,321, ARI = 0,054, NMI = 0,057, tỉ lệ vỡ nợ từng cụm tương quan 0,995 với train, chênh lệch lớn nhất 3,2 điểm % ở cụm nhỏ nhất (cụm 3, 256 mẫu), nên cụm ổn định khi áp lên dữ liệu mới. So sánh: K-Means k = 2 gần như không liên quan nhãn (V = 0,042); K-Means k = 5 và k = 4 có V cao hơn GMM (0,350 và 0,356; ARI 0,044 và 0,079), nên V cao hơn không đồng nghĩa với ARI cao.

## 5. So sánh GMM với K-Means

ARI giữa hai cách gán cụm (1 = trùng hoàn toàn, 0 = như ngẫu nhiên).

In [11]:
ari_tab = pd.DataFrame({'ARI so với GMM (train)': {n: cl.agreement_ari(c_tr[G], c_tr[n]) for n in models if n != G},
                        'ARI so với GMM (validation)': {n: cl.agreement_ari(c_va[G], c_va[n]) for n in models if n != G}})
ari_tab.round(4)

,ARI so với GMM (train),ARI so với GMM (validation)
K-Means k=2 (silhouette),0.2674,0.2756
K-Means k=5 (khuỷu),0.3040,0.3002
K-Means k=4 (= GMM),0.3858,0.3763


**Nhận xét.** ARI giữa GMM k = 4 và K-Means là 0,267 (k = 2), 0,304 (k = 5), 0,386 (k = 4): hai thuật toán chia dữ liệu **giống nhau một phần** nhưng không trùng, kể cả khi cùng k = 4 (ARI = 0,386). Trên validation các giá trị gần y hệt train (0,276 / 0,300 / 0,376), nên khác biệt này là của thuật toán, không phải do nhiễu mẫu.

## 6. Hồ sơ cụm GMM

Trung bình thuộc tính **gốc** (sau `clean` + `add_derived_features`, trước mã hóa/scale/PCA) theo cụm trên train. `BILL_mean`, `PAYAMT_mean` là trung bình 6 tháng (NT$); `nữ`, `sau_ĐH`, `đại_học`, `có_gia_đình` là tỉ lệ (SEX = 2, EDUCATION = 1 / 2, MARRIAGE = 1).

In [12]:
prof = cl.cluster_profile(df_tr, c_tr[G])
prof['default_rate'] = list(rep_gmm['table']['default_rate']) + [y_tr.mean()]
prof.round(2)

,LIMIT_BAL,AGE,PAY_1,BILL_mean,PAYAMT_mean,N_LATE,UTIL_RATIO,nữ,sau_ĐH,đại_học,có_gia_đình,size,default_rate
cụm,,,,,,,,,,,,,
0,212855.83,35.96,-0.63,12803.08,6240.25,0.32,0.11,0.63,0.48,0.40,0.48,8698,0.17
1,166418.69,35.27,0.00,82876.35,5209.91,0.00,0.66,0.60,0.30,0.53,0.44,6381,0.10
2,96400.00,34.76,0.98,53662.44,3596.67,2.70,0.66,0.56,0.24,0.54,0.45,5315,0.45
3,148151.46,38.62,-0.07,46457.70,6381.99,0.51,0.50,0.65,0.07,0.20,0.28,581,0.14
Tất cả,167426.92,35.52,-0.02,45406.57,5260.85,0.83,0.43,0.61,0.35,0.47,0.46,20975,0.22


**Mô tả từng cụm (một câu mỗi cụm, theo số trong bảng).**

- **Cụm 0** (41,5%, vỡ nợ 17,1%): trả đủ và dư nợ thấp: PAY_1 −0,63, BILL_mean 12 803, UTIL_RATIO 0,11, N_LATE 0,32, hạn mức cao nhất (212 856).
- **Cụm 1** (30,4%, 10,5%): 99% dòng mọi PAY = 0 (N_LATE = 0), dư nợ cao nhất (BILL_mean 82 876, UTIL_RATIO 0,66) nhưng trả đúng hạn: vỡ nợ thấp nhất.
- **Cụm 2** (25,3%, 45,3%): trễ hạn thường xuyên: PAY_1 0,98, N_LATE 2,70, hạn mức thấp nhất (96 400), PAYAMT_mean thấp nhất (3 597), UTIL_RATIO 0,66: rủi ro cao nhất.
- **Cụm 3** (2,8%, 13,6%): cụm nhỏ nhất (581 mẫu), tuổi cao nhất (38,6), trình độ thấp hơn (sau ĐH 0,07, đại học 0,20), ít có gia đình (0,28), PAY_1 −0,07, N_LATE 0,51.

**Nhận xét chung.** Cụm tách chủ yếu theo **hành vi trả nợ** (PAY_1, N_LATE) và mức dư nợ, không theo nhân khẩu: tỉ lệ nữ, sau ĐH, đại học, có gia đình ở cụm 0–2 chênh lệch nhỏ (nữ 0,56–0,63; có gia đình 0,44–0,48). Hành vi trả nợ gắn rõ với vỡ nợ: cụm 2 (N_LATE 2,70) vỡ nợ 45,3% so với 10,5% ở cụm 1 (N_LATE 0), dù dư nợ hai cụm tương đương (UTIL_RATIO 0,66). Cụm 3 là cụm khác biệt về hồ sơ (trình độ, tuổi, hôn nhân) nhưng nhỏ (2,8%), nên chưa đủ căn cứ để coi là một phân khúc nhân khẩu rõ ràng. Lưu ý: cụm 1 (mọi PAY = 0) gắn với giá trị PAY = 0 chiếm 33% dòng (xem mục 3b): nhóm dùng tín dụng quay vòng là hành vi có thật, còn hình dạng rất hẹp của cụm (phương sai gần 0) là do điểm khối PAY = 0. Các cụm tách theo giới/trình độ/hôn nhân xuất hiện khi GMM suy biến k lớn (kể cả với `reg_covar = 1e-2`) là nhân tạo do hướng rời rạc trong PCA, không phải phân khúc thật.

## 7. Trực quan PC1–PC2 (màu = cụm, hình = nhãn)

In [13]:
cl.plot_clusters(Z_tr, c_tr[G], y_tr, FIG / 'cluster_gmm_pc12.png', title=f'GMM k={k_gmm} – train: màu = cụm, hình = nhãn')
cl.plot_clusters(Z_tr, c_tr[f'K-Means k={k_sil} (silhouette)'], y_tr, FIG / 'cluster_kmeans_pc12.png',
                 title=f'K-Means k={k_sil} – train: màu = cụm, hình = nhãn')
cl.plot_clusters(Z_tr, c_tr[f'K-Means k={k_gmm} (= GMM)'], y_tr, FIG / 'cluster_kmeans_k_gmm_pc12.png',
                 title=f'K-Means k={k_gmm} – train: màu = cụm, hình = nhãn')
cl.plot_cluster_default_rate(rep_gmm['table'], y_tr.mean(), FIG / 'cluster_gmm_default_rate.png',
                             title=f'GMM k={k_gmm} – tỉ lệ vỡ nợ theo cụm (train)')
print('đã lưu 4 hình')

đã lưu 4 hình


![](../results/figures/cluster_gmm_pc12.png)

![](../results/figures/cluster_kmeans_pc12.png)

![](../results/figures/cluster_kmeans_k_gmm_pc12.png)

![](../results/figures/cluster_gmm_default_rate.png)

**Nhận xét.** Trên PC1–PC2, 4 cụm GMM tách được phần nào: cụm 2 (xanh lá, vỡ nợ cao) nằm ở PC2 cao và PC1 dương, cụm 1 (cam) là đám dày ở PC1 ≈ 1–3, PC2 ≈ −2–0 chồng lên cụm 2 ở rìa, cụm 0 (xanh dương) rải rộng về phía PC1 âm; cụm 3 (tím, 2,8%) lẫn vào các cụm khác trên hai chiều này. Điểm vỡ nợ (tam giác) tập trung ở PC2 cao, cùng chỗ với cụm 2, khớp tỉ lệ vỡ nợ 45,3% của cụm. K-Means k = 2 cắt gần như thẳng đứng tại PC1 ≈ −1 (trái/phải), hai bên có tỉ lệ vỡ nợ gần nhau (23,3% và 19,5%), nên cách chia này không phân biệt nhãn.

## 8. Lưu mô hình để tái lập

Lưu preprocessor, PCA, GMM, K-Means vào `data/models/` (gitignore) để tái lập mục 4.3. `predict_proba_df` nạp lại và cho P(cụm | x); kiểm tra khớp với `gmm.predict_proba` và tổng mỗi dòng bằng 1. Mô-đun 2 tầng (mục 5) **không dùng** mô hình này: nó fit GMM riêng trên khối A (LIMIT_BAL, SEX, EDUCATION, MARRIAGE, AGE), dùng lại `select_k_gmm` / `fit_gmm` / `save_models`. Posterior của GMM gần như cứng (max P trung bình in ra bên dưới), nên P(cụm | x) gần một one-hot.

In [14]:
paths = cl.save_models('cluster_7-3_seed0', pre, pca, gmm, km_main)
loaded = cl.load_models('cluster_7-3_seed0')
proba_va = cl.predict_proba_df(loaded, df_va)
assert np.allclose(proba_va.to_numpy(), gmm.predict_proba(Z_va)) and np.allclose(proba_va.sum(axis=1), 1)
print({k: p.name for k, p in paths.items()})
print('P(cụm | x) validation:', proba_va.shape, '| độ chắc chắn trung bình (max P) =', round(proba_va.max(axis=1).mean(), 3))
print(f'Tổng thời gian chạy: {time.time() - T0:.0f} s')
proba_va.head(3).round(3)

{'pre': 'cluster_7-3_seed0_pre.joblib', 'pca': 'cluster_7-3_seed0_pca.joblib', 'gmm': 'cluster_7-3_seed0_gmm.joblib', 'kmeans': 'cluster_7-3_seed0_kmeans.joblib'}
P(cụm | x) validation: (8990, 4) | độ chắc chắn trung bình (max P) = 0.973
Tổng thời gian chạy: 266 s


,0,1,2,3
6292,0.0,0.005,0.932,0.064
10828,1.0,0.000,0.000,0.000
27020,0.0,0.000,0.000,1.000
